# Week 1 · Lab: Tokens, context, and embeddings

**The LLM Resident** · the lab for *Tokens, Context, and Embeddings*

One word, two meanings. In these four sentences, "bank" is either the side of a river or a place that holds money:

- A: They fished from the muddy bank of the river.
- B: She sat on the river bank all afternoon.
- C: The investment bank approved the merger.
- D: He deposited the check at the bank.

The question this lab answers: does that one word enter the model differently depending on the sentence it sits in? We measure three things. First, how the tokenizer turns text into integer ids, and whether "bank" gets the same id in every sentence. Second, how the vector for "bank" changes as it moves up through the layers of GPT-2 and BERT. Third, how a sentence embedding model places whole sentences so a query can retrieve the right ones.

Run every cell top to bottom (`Shift+Enter`). The `assert` lines are self-checks: if a cell runs silently, its basic invariants hold. The first run downloads about 1 GB of model weights from the Hugging Face Hub and caches them locally. Everything runs on CPU; no GPU is needed.

## 1 · Set up the environment

This lab uses three libraries: `torch` for the tensors, `transformers` for GPT-2 and BERT, and `sentence-transformers` for the sentence embedding model. The cell below installs them only if they are missing, so it runs silently where they are already present (for example the environment this lab was validated in) and installs them on first use in Colab. The pinned versions are listed at the end.

In [ ]:
# Colab and fresh environments install the pinned dependencies on first run.
# Where they are already present, this cell just confirms the imports resolve.
import importlib.util
import sys

DEPENDENCIES = {"torch": "torch==2.13.0", "transformers": "transformers==5.14.1", "sentence_transformers": "sentence-transformers==5.6.1"}
missing = [pip_spec for module, pip_spec in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
if missing:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

from importlib.metadata import version
print("python", sys.version.split()[0])
for module, pip_spec in DEPENDENCIES.items():
    name = pip_spec.split("==")[0]
    print(name, version(name))

## 2 · From text to integers

A model never sees letters. The tokenizer splits each sentence into tokens and maps every token to an integer id. We use the GPT-2 tokenizer here. GPT-2 uses byte-level BPE, so a leading space is part of the token: the `Ġ` marker you will see stands for that space. Watch what happens to "bank". It carries a leading space in all four sentences, so it becomes the same token, and therefore the same id, every time. The sense of the word is nowhere in this id. That is the point: meaning has to come from somewhere else.

In [ ]:
from transformers import AutoTokenizer

SENTS = {
    "A": "They fished from the muddy bank of the river.",
    "B": "She sat on the river bank all afternoon.",
    "C": "The investment bank approved the merger.",
    "D": "He deposited the check at the bank.",
}
SENSE = {"A": "river", "B": "river", "C": "finance", "D": "finance"}

# Pin the exact model snapshots so the numbers cannot drift under this notebook.
REVISIONS = {
    "gpt2": "607a30d783dfa663caf39e06633721c8d4cfcd7e",
    "bert-base-uncased": "86b5e0934494bd15c9632b12f734a8a67f723594",
    "sentence-transformers/all-MiniLM-L6-v2": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
}

gpt2_tok = AutoTokenizer.from_pretrained("gpt2", revision=REVISIONS["gpt2"])

def bank_position(tokens):
    """Index of the single token whose surface form is 'bank', ignoring GPT-2's space marker and BERT's ## prefix."""
    hits = [i for i, t in enumerate(tokens) if t.lstrip("\u0120").lstrip("#").lower() == "bank"]
    assert len(hits) == 1, tokens
    return hits[0]

for k, sentence in SENTS.items():
    ids = gpt2_tok.encode(sentence)
    tokens = gpt2_tok.convert_ids_to_tokens(ids)
    i = bank_position(tokens)
    print(f"{k} ({SENSE[k]:7s}) bank id = {ids[i]:5d}   tokens = {tokens}")

In [ ]:
# self-check: the id for "bank" is stable, and the leading space changes it
assert gpt2_tok.encode(" bank", add_special_tokens=False) == [3331], "expected ' bank' -> [3331]"
assert gpt2_tok.encode("bank", add_special_tokens=False) == [17796], "expected 'bank' -> [17796]"

bank_ids = set()
for sentence in SENTS.values():
    ids = gpt2_tok.encode(sentence)
    bank_ids.add(ids[bank_position(gpt2_tok.convert_ids_to_tokens(ids))])
assert bank_ids == {3331}, f"expected the same bank id in all four sentences, got {bank_ids}"

print("checks passed: ' bank' -> 3331, 'bank' -> 17796, and every sentence uses id 3331 for bank")

## 3 · Watch the vector for "bank" change through the layers

The id is the same everywhere, so the model has to build meaning as the token moves up through its layers. Each layer reads the whole sentence and rewrites every token's vector in light of its neighbors. We take the vector sitting at the "bank" position and read it off at three depths: layer 0 (the input, before any attention), a middle layer, and the final layer. Then we measure cosine similarity between pairs. Sentences A and B both mean the river; C and D both mean the money. If the model is learning the sense, the same-sense pairs should pull together and the cross-sense pairs should push apart as we climb.

In [ ]:
import torch
from transformers import AutoModel

torch.set_grad_enabled(False)

PAIRS = [("A", "B", "same-sense (river)"), ("C", "D", "same-sense (finance)"),
         ("A", "C", "cross-sense"), ("A", "D", "cross-sense"),
         ("B", "C", "cross-sense"), ("B", "D", "cross-sense")]

def cosine(a, b):
    return float(torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)))

def bank_vectors_by_layer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name, revision=REVISIONS[model_name])
    model = AutoModel.from_pretrained(model_name, revision=REVISIONS[model_name],
                                      output_hidden_states=True)
    model.eval()
    hidden = {}
    for k, sentence in SENTS.items():
        enc = tok(sentence, return_tensors="pt")
        tokens = tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())
        i = bank_position(tokens)
        states = model(**enc).hidden_states   # tuple: input embeddings, then one entry per layer
        hidden[k] = {L: states[L][0, i] for L in range(len(states))}
    n_layers = len(hidden["A"]) - 1
    return hidden, n_layers

def similarity_table(model_name):
    hidden, n_layers = bank_vectors_by_layer(model_name)
    layers = sorted({0, n_layers // 2, n_layers})
    print(f"{model_name}: {n_layers} layers, cosine similarity of the 'bank' vector at layers {layers}")
    print(f"{'pair':<5}{'kind':<22}" + "".join(f"{'L' + str(L):<9}" for L in layers))
    for x, y, kind in PAIRS:
        row = "".join(f"{cosine(hidden[x][L], hidden[y][L]):<9.3f}" for L in layers)
        print(f"{x + '-' + y:<5}{kind:<22}{row}")
    print("mean cosine by layer (same-sense | cross-sense), the full curves:")
    for L in range(n_layers + 1):
        s = (cosine(hidden["A"][L], hidden["B"][L]) + cosine(hidden["C"][L], hidden["D"][L])) / 2
        c = sum(cosine(hidden[x][L], hidden[y][L]) for x, y, kind in PAIRS if kind == "cross-sense") / 4
        print(f"  L{L:<3} {s:.4f} | {c:.4f}")
    same = [cosine(hidden["A"][n_layers], hidden["B"][n_layers]), cosine(hidden["C"][n_layers], hidden["D"][n_layers])]
    cross = [cosine(hidden[x][n_layers], hidden[y][n_layers]) for x, y, kind in PAIRS if kind == "cross-sense"]
    same_mean, cross_mean = sum(same) / len(same), sum(cross) / len(cross)
    print(f"final layer L{n_layers}: mean same-sense = {same_mean:.3f}, mean cross-sense = {cross_mean:.3f}")
    return same_mean, cross_mean

print("BERT (bert-base-uncased)")
bert_same, bert_cross = similarity_table("bert-base-uncased")
print()
print("GPT-2 (gpt2)")
gpt2_same, gpt2_cross = similarity_table("gpt2")

In [ ]:
# self-check (BERT): same-sense vectors are markedly closer than cross-sense at the final layer
assert bert_same - bert_cross >= 0.15, f"expected BERT same-sense to lead cross-sense by >= 0.15, got {bert_same - bert_cross:.3f}"
print(f"BERT  final layer: same-sense {bert_same:.3f} vs cross-sense {bert_cross:.3f}  (gap {bert_same - bert_cross:.3f})")

# GPT-2 is reported, not asserted: its final layer shows almost no separation.
print(f"GPT-2 final layer: same-sense {gpt2_same:.3f} vs cross-sense {gpt2_cross:.3f}  (gap {gpt2_same - gpt2_cross:.3f})")

BERT separates the two senses cleanly by its final layer: same-sense pairs sit near 0.74 and cross-sense pairs near 0.45. GPT-2 does not. Its final-layer vectors for "bank" are around 0.98 apart whether the senses match or not, which is why the gap is printed rather than asserted. Two things are worth naming. GPT-2's late layers are anisotropic: the vectors crowd into a narrow cone, so almost everything looks similar under cosine and the raw number stops being a clean sense signal. And GPT-2 is trained only to predict the next token, so its final vector at a position is shaped to guess what follows, not to be a tidy summary of the current word's meaning. The essay works through both points.

Look at layer 0 as well. Where "bank" sits at the same position in two sentences, its layer-0 cosine is exactly 1.0, even across senses (GPT-2 A vs D, BERT A vs B). Layer 0 is just the token embedding plus the position embedding, and both are identical there, so the vectors are identical before any attention has run. Everything that separates the senses is built by the layers above.

## 4 · Is the sense information still in GPT-2?

The cosine probe on GPT-2's final-layer hidden states showed no sense separation: same-sense and cross-sense pairs both sat near 0.98. That could mean one of two things. Either GPT-2 has genuinely lost track of which "bank" it is reading, or the information is still there and cosine similarity is the wrong instrument to see it. So this cell asks GPT-2 the one question it was actually trained to answer: given the sentence up to and including "bank", which token comes next? We cut each sentence off right after the " bank" token (id 3331), run the language-model head, softmax the final-position logits, and read the top-5 next tokens. If the predictions differ by sense, the context survived the layers even though the cosine probe could not see it.

In [ ]:
import torch
from transformers import GPT2LMHeadModel

torch.set_grad_enabled(False)
lm = GPT2LMHeadModel.from_pretrained("gpt2", revision=REVISIONS["gpt2"])
lm.eval()

BANK = 3331   # the " bank" token id, shared by all four sentences

next_token_top5 = {}
for k, sentence in SENTS.items():
    ids = gpt2_tok.encode(sentence)
    cut = ids.index(BANK) + 1                       # keep the sentence up to and including " bank"
    prefix_ids = ids[:cut]
    logits = lm(input_ids=torch.tensor([prefix_ids])).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, 5)
    next_token_top5[k] = [(gpt2_tok.decode([i]), float(p)) for p, i in zip(top.values, top.indices)]
    print(f"{k} ({SENSE[k]:7s}) next token after {gpt2_tok.decode(prefix_ids)!r}:")
    for token, p in next_token_top5[k]:
        print(f"    {p:.4f}  {token!r}")
    print()

In [ ]:
# self-check: the next-token predictions carry the sense the cosine probe could not see
c_top3 = [token for token, _ in next_token_top5["C"][:3]]
assert "'s" in c_top3, f"expected 's among the top-3 after the investment-bank prefix, got {c_top3}"

a_top1 = next_token_top5["A"][0][0]
assert a_top1 == " of", f"expected ' of' as the top prediction after the muddy-bank prefix, got {a_top1!r}"

print("checks passed: the finance prefix predicts 's in its top 3, the river prefix predicts ' of' first")

Read the two lists. After "The investment bank", GPT-2's likeliest continuations are "'s", "has", "said", "is": the words that follow an institution about to act. After "the muddy bank", the top continuation is " of", as in "bank of the river", then " and" and a comma. The distributions differ, and they differ in the direction the senses predict. The sense was in the network the whole time. Cosine on a single final-layer vector was just too blunt to reveal it, for the anisotropy and next-token reasons noted above. Choosing the right probe matters as much as running one.

## 5 · Embed whole sentences and retrieve them

The layer experiment gave us one vector per token. For search and retrieval we want one vector per sentence. `all-MiniLM-L6-v2` is a sentence embedding model that maps any text to a single 384-dimensional vector, tuned so texts with similar meaning land close together. We embed the four sentences, then embed two short queries and rank the sentences by cosine similarity to each. A query about finance should surface C and D; a query about a riverbank should surface A and B. Same word "bank" in all four, but the sentence vectors know which is which.

In [ ]:
from sentence_transformers import SentenceTransformer, util

st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                         revision=REVISIONS["sentence-transformers/all-MiniLM-L6-v2"])
labels = list("ABCD")
sent_vectors = st.encode([SENTS[k] for k in labels], convert_to_tensor=True, normalize_embeddings=True)

print("embedding dimension:", sent_vectors.shape[1])
print()
print("pairwise cosine similarity between the four sentences:")
for i, a in enumerate(labels):
    for j, b in enumerate(labels):
        if i < j:
            sim = float(util.cos_sim(sent_vectors[i], sent_vectors[j]))
            print(f"  {a}-{b} ({SENSE[a]:7s}/ {SENSE[b]:7s}): {sim:.3f}")

QUERIES = {"financial institution": "financial institution", "the edge of a river": "the edge of a river"}
rankings = {}
for name, query in QUERIES.items():
    qv = st.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scored = sorted(((k, float(util.cos_sim(qv, sent_vectors[i]))) for i, k in enumerate(labels)), key=lambda t: -t[1])
    rankings[name] = scored
    print()
    print(f"query: {name!r}")
    for k, score in scored:
        print(f"  {k} ({SENSE[k]:7s}): {score:.3f}   {SENTS[k]}")

In [ ]:
# self-check: 384 dimensions, and each query retrieves the sentences of the matching sense first
assert sent_vectors.shape[1] == 384, f"expected 384-dim embeddings, got {sent_vectors.shape[1]}"

def top_two(name):
    return {k for k, _ in rankings[name][:2]}

assert top_two("financial institution") == {"C", "D"}, "both finance sentences should outrank both river sentences for 'financial institution'"
assert top_two("the edge of a river") == {"A", "B"}, "both river sentences should outrank both finance sentences for 'the edge of a river'"

print("checks passed: 384-dim embeddings; finance query surfaces C and D, river query surfaces A and B")

## What you just did

- **Tokenized text into ids.** The tokenizer turned each sentence into integers. "bank" got the same id, 3331, in all four sentences, so the id carries no sense on its own.
- **Watched context do the work.** Reading the "bank" vector at each layer, BERT pulled the same-sense sentences together and pushed the cross-sense ones apart by its final layer (about 0.74 vs 0.45). Under cosine, GPT-2's final layer showed almost no separation, for the reasons the essay covers.
- **Asked the model directly.** Cutting each sentence after "bank" and reading GPT-2's next-token distribution, the two senses predicted different continuations (" of" after the muddy bank, "'s" and "has" after the investment bank). The sense was there all along; cosine on one final-layer vector was the wrong instrument to see it.
- **Embedded whole sentences.** `all-MiniLM-L6-v2` mapped each sentence to one 384-dimensional vector, and those vectors ranked the finance sentences first for a finance query and the river sentences first for a river query. The finance query shares no word with its sentences; the river query shares only "river". The ranking follows meaning, not shared words.

The pipeline you traced, end to end:

`text → tokens → ids → input vectors → contextual vectors → one vector per text`

**Environment.** These numbers were produced on CPU with Python 3.11.15, torch 2.13.0, transformers 5.14.1, and sentence-transformers 5.6.1. The model weights are pinned by revision: gpt2 `607a30d`, bert-base-uncased `86b5e09`. Cosine values can shift slightly across library or hardware versions; the rankings and the separation pattern are what to hold onto.

**The essay:** [Tokens, Context, and Embeddings](https://revanthreddy-hai.github.io/the-llm-residency/week01.html).